# Almacena — Lender Loan Book (exploration)

> ⚠️ **April = source · Jan–Mar = derived · 2026 · USD (with EUR overlay).** The lender file is the April export, so **April is the actual/source month** and the reconstruction reproduces its figures exactly. **Jan–Mar are derived** by replaying each loan's active window. EUR is converted at each month's own average USD/EUR rate (see `loan_book.FX_USD_PER_EUR`; X-Rates cross-checked vs ECB). See `README.md`.

In [ ]:
import sys, os, datetime as dt
sys.path.insert(0, os.getcwd())   # run from loansdb/
import loan_book, pandas as pd
import matplotlib.pyplot as plt
TEAL, ACCENT, CORAL = '#013E3F', '#009091', '#E67D5A'
data = loan_book.build()
loans = loan_book._load_loans()
print(data['derived_note'])
print('FX:', data['fx_source'])
print('scope:', data['scope'], '| loans in file:', len(loans))

## Monthly book (USD, with EUR at each month's rate)

In [ ]:
summary = pd.DataFrame([{'month':m['label'],'status':'source' if m['is_source'] else 'derived',
    'available_$':m['available'],'available_EUR':round(m['available']/m['fx_usd_per_eur']),
    'cost_$':m['cost'],'blended_%':round(m['blended_rate']*100,2),
    'n_loans':m['n_loans'],'new':m['n_new'],'maturing':m['n_maturing'],
    'usd_per_eur':m['fx_usd_per_eur']} for m in data['months']]).set_index('month')
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(summary.index, summary['available_$']/1e6, color=TEAL, label='Available ($M)')
ax2 = ax.twinx()
ax2.plot(summary.index, summary['cost_$']/1e3, color=CORAL, marker='o', lw=2, label='Cost ($k)')
ax.set_ylabel('Available funds  $M'); ax2.set_ylabel('Cost of funds  $k')
ax.set_title('Lender funding book — 2026 (Apr source, Jan-Mar derived; USD)')
plt.tight_layout(); plt.show()

## Month-over-month change (USD)

In [ ]:
mom = pd.DataFrame([{'month':m['label'], **(m['mom'] or {})} for m in data['months']]).set_index('month')
mom

## Lender concentration — April

In [ ]:
apr = loan_book.month_book(loans, 2026, 4)
ldf = pd.DataFrame(apr['loans'])
conc = ldf.groupby('lender')['available'].sum().sort_values(ascending=False)
ax = conc.head(8).iloc[::-1].plot.barh(color=ACCENT, figsize=(8,4))
ax.set_title('Top lenders by available funds — Apr 2026 (USD)'); ax.set_xlabel('USD')
plt.tight_layout(); plt.show()
conc.head(8)

## Funding maturity wall (forward-looking, from the schedule)

In [ ]:
mat = {}
for ln in loans:
    if ln['repay'] >= dt.date(2026,4,1):
        k = ln['repay'].strftime('%Y-%m'); mat[k] = mat.get(k,0) + ln['principal']
mser = pd.Series(mat).sort_index()
ax = mser.div(1e6).plot.bar(color=TEAL, figsize=(9,4))
ax.set_title('Principal maturing by month  ($M, DERIVED)'); ax.set_ylabel('$M')
plt.tight_layout(); plt.show()
mser

## Rate distribution — April active book

In [ ]:
ax = ldf['rate'].mul(100).plot.hist(bins=10, color=CORAL, edgecolor='white', figsize=(7,3.5))
ax.set_xlabel('Annual rate  %'); ax.set_title('April active-loan rate distribution')
plt.tight_layout(); plt.show()
print('blended (principal-weighted):', round(apr['blended_rate']*100,2), '%')

---
*April is source; Jan–Mar are derived snapshots for analysis. EUR at real monthly rates (differs from the deck's held 1.087). To use real monthly data, point `loan_book.py` at per-month files.*